In [1]:
# Desinstalar para limpiar
!pip uninstall -y torch torchtext torchdata


Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchdata 0.11.0
Uninstalling torchdata-0.11.0:
  Successfully uninstalled torchdata-0.11.0


In [2]:
%%capture
!pip install torch==2.2.2 torchtext==0.17.2 torchdata==0.7.1 portalocker>=2.0.0

In [3]:
!pip uninstall -y numpy

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2


In [1]:
!pip install numpy==1.26.4

In [2]:
import torch
import torchtext
from torchtext.datasets import AG_NEWS
import portalocker

print(f"Versión Torch: {torch.__version__}")
print(f"Versión Torchtext: {torchtext.__version__}")

# Prueba rápida para ver si AG_NEWS responde
try:
    train_iter = iter(AG_NEWS(split="train"))
    print("¡Éxito! El dataset está listo.")
except Exception as e:
    print(f"Error al cargar dataset: {e}")

Versión Torch: 2.2.2+cu121
Versión Torchtext: 0.17.2+cpu
¡Éxito! El dataset está listo.


In [3]:
next(train_iter)

(3,
 "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.")

In [4]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

tokenizador = get_tokenizer('spacy' , language="en_core_web_sm")
train_iter = AG_NEWS(split='train')
train_iter = iter(train_iter)

def yield_tokens(data_iter):

  for _, text in data_iter:

    yield tokenizador(text)



In [5]:
vocab = build_vocab_from_iterator(yield_tokens(train_iter) , specials=['<unk>'])
vocab.set_default_index(vocab['<unk>'])


In [6]:
tokenizador("Hello how are you ? I am a engineer system student")

['Hello',
 'how',
 'are',
 'you',
 '?',
 'I',
 'am',
 'a',
 'engineer',
 'system',
 'student']

In [7]:
vocab(tokenizador("Hello how are you ? I am a engineer system student"))

[18574, 446, 45, 191, 134, 310, 3191, 6, 4681, 279, 4369]

In [8]:
# Pipelines: transforman texto a IDs y etiquetas a base 0
text_pipeline = lambda x: vocab(tokenizador(x))
label_pipeline = lambda x: int(x) - 1

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [10]:
def collate_batch(batch):
    label_list, text_list, offsets = [], [], [0]
    for (_label, _text) in batch:
         label_list.append(label_pipeline(_label))
         processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
         text_list.append(processed_text)
         offsets.append(processed_text.size(0)) # Guardamos el largo de cada noticia

    label_list = torch.tensor(label_list, dtype=torch.int64)
    # Convertimos los largos en posiciones acumuladas (necesario para EmbeddingBag)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)
    return label_list.to(device), text_list.to(device), offsets.to(device)

In [11]:
from torch.utils.data import DataLoader


train_iter = AG_NEWS(split='train')
dataloader = DataLoader(train_iter, batch_size=10, shuffle=True, collate_fn=collate_batch)

In [12]:
import torchtext.vocab as vocab_vectors
#Descargamos GloVe
glove = vocab_vectors.GloVe(name='6B', dim=100)

vocab_size = len(vocab)
embed_dim = 100
weights_matrix = torch.zeros((vocab_size, embed_dim))

for i, word in enumerate(vocab.get_itos()):
    if word in glove.stoi:
        weights_matrix[i] = glove.vectors[glove.stoi[word]]
    else:
        # Si la palabra no está en GloVe, le asignamos un vector aleatorio
        weights_matrix[i] = torch.randn(embed_dim)

.vector_cache/glove.6B.zip: 862MB [02:41, 5.34MB/s]                           
100%|█████████▉| 399999/400000 [00:18<00:00, 21439.78it/s]


In [13]:
import torch.nn as nn
import torch.nn.functional as F

class ModelTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class, pretrained_weights=None):
        super(ModelTextClassifier, self).__init__()
        # Mantenemos el EmbeddingBag para eficiencia
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights) # Cargamos GloVe

        # Capa oculta extra
        self.fc1 = nn.Linear(embed_dim, 64)
        self.dropout = nn.Dropout(0.3) # Apaga el 30% de neuronas al azar para generalizar mejor
        self.fc2 = nn.Linear(64, num_class)

        self.init_weights()

    def init_weights(self):
        initrange = 0.5
        self.fc1.weight.data.uniform_(-initrange, initrange)
        self.fc2.weight.data.uniform_(-initrange, initrange)
        self.fc1.bias.data.zero_()
        self.fc2.bias.data.zero_()

    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        x = F.relu(self.fc1(embedded)) # Activación ReLU para aprender patrones no lineales
        x = self.dropout(x)
        return self.fc2(x)

In [14]:
train_iter = AG_NEWS(split='train')
num_class = len(set([label for (label, text) in AG_NEWS(split='train')]))
# Usamos 100 porque es la dimensión de GloVe que elegimos
modelo = ModelTextClassifier(vocab_size, 100, num_class, weights_matrix).to(device)

# Definimos el optimizador y la pérdida (como en tu notebook original)
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

In [15]:
vocab_size

110933

In [16]:
modelo

ModelTextClassifier(
  (embedding): EmbeddingBag(110933, 100, mode='mean')
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=4, bias=True)
)

In [17]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'El modelo tiene {count_parameters(modelo):,} ')

El modelo tiene 11,100,024 


In [18]:
import time

def entrenamiento(dataloader, model, optimizer, criterion):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 500
    start_time = time.time()

    for idx, (label, text, offsets) in enumerate(dataloader):
        optimizer.zero_grad()
        predicted_label = model(text, offsets)
        loss = criterion(predicted_label, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1) # Clip de gradientes
        optimizer.step()

        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)

        if idx % log_interval == 0 and idx > 0:
            elapsed = time.time() - start_time
            print(f'| época {epoch:3d} | {idx:5d}/{len(dataloader):5d} batches '
                  f'| precisión {total_acc/total_count:8.3f}')

    return total_acc / total_count

In [19]:
def evalua(dataloader, model, criterion):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        for idx, (label, text, offsets) in enumerate(dataloader):
            predicted_label = model(text, offsets)
            loss = criterion(predicted_label, label)
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0) # CORREGIDO: antes decía total_account

    return total_acc / total_count

In [21]:
from torch.utils.data.dataset import random_split
from torchtext.data.functional import to_map_style_dataset

train_iter , test_iter = AG_NEWS()

x_train = to_map_style_dataset(train_iter)
x_test = to_map_style_dataset(test_iter)

batch_size = 34

num_train = int(len(x_train) * 0.95 )

split_train , split_valid = random_split(x_train , [num_train , len(x_train) - num_train])

train_dataloader = DataLoader(split_train, batch_size=batch_size, shuffle = True , collate_fn = collate_batch)
valid_dataloader = DataLoader(split_valid, batch_size=batch_size, shuffle = True , collate_fn = collate_batch)
test_dataloader =  DataLoader(x_test     , batch_size=batch_size, shuffle = True , collate_fn = collate_batch)


In [22]:
# Definir hiperparámetros básicos si no los tienes
epochs = 10
best_valid_acc = 0

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    # CORREGIDO: Pasamos todos los argumentos necesarios
    train_acc = entrenamiento(train_dataloader, modelo, optimizer, criterion)
    valid_acc = evalua(valid_dataloader, modelo, criterion)

    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        # Sugerencia: Guardar el mejor modelo
        torch.save(modelo.state_dict(), 'mejor_modelo.pt')

    print('-' * 59)
    print(f'| fin de época {epoch:3d} | tiempo: {time.time() - epoch_start_time:5.2f}s | '
          f'precisión validación {valid_acc:8.3f} ')
    print('-' * 59)

| época   1 |   500/ 3353 batches | precisión    0.693
| época   1 |  1000/ 3353 batches | precisión    0.765
| época   1 |  1500/ 3353 batches | precisión    0.800
| época   1 |  2000/ 3353 batches | precisión    0.821
| época   1 |  2500/ 3353 batches | precisión    0.835
| época   1 |  3000/ 3353 batches | precisión    0.845
-----------------------------------------------------------
| fin de época   1 | tiempo: 55.57s | precisión validación    0.903 
-----------------------------------------------------------
| época   2 |   500/ 3353 batches | precisión    0.923
| época   2 |  1000/ 3353 batches | precisión    0.923
| época   2 |  1500/ 3353 batches | precisión    0.923
| época   2 |  2000/ 3353 batches | precisión    0.924
| época   2 |  2500/ 3353 batches | precisión    0.925
| época   2 |  3000/ 3353 batches | precisión    0.926
-----------------------------------------------------------
| fin de época   2 | tiempo: 53.96s | precisión validación    0.913 
----------------------

In [23]:
# 1. (Opcional) Cargamos los mejores pesos guardados durante el entrenamiento
# Esto asegura que evaluamos la mejor versión del modelo, no solo la de la última época.
try:
    modelo.load_state_dict(torch.load('mejor_modelo.pt'))
    print("¡Pesos del mejor modelo cargados con éxito!")
except:
    print("Usando los pesos actuales del modelo.")

# 2. Ejecutamos la evaluación sobre el test_dataloader
test_acc = evalua(test_dataloader, modelo, criterion)

print(f'\n' + '='*30)
print(f'RESULTADO FINAL EN TEST')
print(f'Precisión (Accuracy): {test_acc * 100:.2f}%')
print('='*30)

¡Pesos del mejor modelo cargados con éxito!

RESULTADO FINAL EN TEST
Precisión (Accuracy): 92.00%


In [27]:
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

def predict(text, text_pipeline):
    with torch.no_grad():
        text_tensor = torch.tensor(text_pipeline(text), dtype=torch.int64).to(device)
        # Como es una sola noticia, el offset es 0
        offsets = [0]
        offsets = torch.tensor(offsets).to(device)
        output = modelo(text_tensor, offsets)
        return output.argmax(1).item() + 1 # +1 porque restamos 1 en el pipeline

# Ejemplo de uso:
print(f"La noticia es: {ag_news_label[predict("Real Madrid vs Barcelona", text_pipeline)]}")

La noticia es: Sports
